In [86]:
from collections.abc import Callable

def apply_rule(text: str, rule: Callable[[str], bool]) -> bool:
    return rule(text)

def check_length(text: str) -> bool:
    return len(text) > 10

def contain_keyword(text: str) -> bool:
    return "def" in text.lower()

l = apply_rule("这是检测长度的规则 length", check_length)
assert l == True
l2 = apply_rule("这是检测长度的规则", check_length)
assert l2 == False

k = apply_rule("这是检测包含关键字Def的规abc则", contain_keyword)
assert k == True
k2 = apply_rule("这是检测包含关键字cef的规abc则", contain_keyword)
assert k2 == False



In [19]:
from functools import wraps


def count_calls(fn):
    @wraps(fn)
    def w(*args, **kwargs):
        w.calls += 1
        return fn(*args, **kwargs)
    w.calls = 0
    return w

@count_calls
def count():
    return count.calls

assert count.__name__ == "count"
assert count() == 1
assert count() == 2
assert count() == 3

In [80]:
from collections.abc import Iterator
import pytest

def iter_chunks(text: str, chunk_size: int) -> Iterator[str]:
    if len(text) <= 0:
        raise ValueError("text不能为空")
    if chunk_size <= 0:
        raise ValueError("非法的chunk_size")
    for i in range(0, len(text), chunk_size):
        yield text[i : i + chunk_size]


iter2 = iter_chunks("abcdefgh", 3)
assert next(iter2) == "abc"
assert next(iter2) == "def"
assert next(iter2) == "gh"

with pytest.raises(ValueError) as exc_info:
    iter3 = iter_chunks("", 3)
    next(iter3)
assert "text" in str(exc_info.value)

with pytest.raises(ValueError) as exc_info:
    iter3 = iter_chunks("abcdefgh", 0)
    next(iter3)
assert "chunk_size" in str(exc_info.value)

with pytest.raises(ValueError) as exc_info:
    iter3 = iter_chunks("abcdefgh", -1)
    next(iter3)
assert "chunk_size" in str(exc_info.value)


In [84]:
from pydantic import BaseModel, Field
from dataclasses import dataclass, field
import pytest


@dataclass(kw_only=True)
class Document:
    title: str
    content: str
    category: str
    tags: list[str]
    source: str
    word_count: int = field(init=False)

    def __post_init__(self) -> None:
        self.word_count = len(self.content)


class DocumentCreate(BaseModel):
    title: str = Field(min_length=2, max_length=20)
    content: str = Field(min_length=1)
    category: str
    tags: list[str] = Field(default_factory=list, max_length=10)
    source: str


class InvalidTitleError(ValueError):
    """包含违禁关键词"""

    def __init__(self, title: str, reason: str, forbidden_word: str) -> None:
        self.title = title
        self.reason = reason
        self.forbidden_word = forbidden_word
        super().__init__(f"{reason}，标题“{title}”存在违禁词：{forbidden_word}")


def create_document(payload: DocumentCreate) -> Document:
    forbidden = ["测试", "课件", "广告"]
    for v in forbidden:
        if v in payload.title:
            raise InvalidTitleError(title=payload.title, reason="不能包含违禁词", forbidden_word=v)
    document = Document(**payload.model_dump())
    return document


@pytest.fixture
def payload():
    return {'title': "文档标题1", 'content': "文档内容1", 'category': "分类1", 'tags': ["tag1", "tag2"],
            'source': "renrendoc"}


def test_create_document(payload):
    dc = DocumentCreate(**payload)
    document = create_document(dc)
    result = {**payload, 'word_count': 5}
    assert document.__dict__ == result


@pytest.mark.parametrize("title, exception, match", [
    ("文档标题1测试", InvalidTitleError, "存在违禁词：测试"),
    ("文档广告标题1", InvalidTitleError, "存在违禁词：广告"),
    ("课件文档标题1", InvalidTitleError, "存在违禁词：课件")
])
def test_title_forbidden_test(payload, title, exception, match):
    payload['title'] = title
    with pytest.raises(exception, match=match):
        list(create_document(DocumentCreate(**payload)))